Generate a feature indicating how long a customer has been a member by computing membership duration dynamically, normalizing the feature using log transformation, and segmenting customers into lifecycle stages based on engagement trends.

In [2]:
import pandas as pd

df_shoppers = pd.read_csv("online_shoppers_intention.csv")

print("Shape:", df_shoppers.shape)
print(df_shoppers.columns.tolist())
display(df_shoppers.head())

Shape: (12330, 18)
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


Create engagement and lifecycle features

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [5]:
task24_df = df_shoppers.copy()

# 1. Calculate total page visits
task24_df['Total_Page_Visits'] = (task24_df['Administrative'] + task24_df['Informational'] + + task24_df['ProductRelated'])

# 2. Calculate total time spent on website
task24_df['Total_Time_Spent'] = (task24_df['Administrative_Duration'] + task24_df["Informational_Duration"] + task24_df["ProductRelated_Duration"]) 

# 3. Log-transform engagement features
task24_df['Log_Total_Page_Visits'] = np.log1p(task24_df['Total_Page_Visits'])

task24_df["Log_Total_Time_Spent"] = np.log1p(task24_df["Total_Time_Spent"])

# 4. Normalize transformed features
scaler = MinMaxScaler()

task24_df[
    ["Log_Visits_Normalized", "Log_Time_Normalized"]
] = scaler.fit_transform(
    task24_df[
        ["Log_Total_Page_Visits", "Log_Total_Time_Spent"]
    ]
)

# 5. Create an engagement score
task24_df['Engagement_Score'] = (0.5 * task24_df['Log_Visits_Normalized'] + 0.5 * task24_df['Log_Time_Normalized'])

# 6. Cluster sessions into engagement groups
engagement_features = task24_df[['Log_Visits_Normalized', 'Log_Time_Normalized']]

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

task24_df['Engagement_Cluster'] = kmeans.fit_predict(engagement_features)

# Rank clusters by average engagement score
cluster_order = (
    task24_df.groupby("Engagement_Cluster")["Engagement_Score"]
    .mean()
    .sort_values()
    .index
)


lifecycle_mapping = {
    cluster: stage
    for cluster, stage in zip(
        cluster_order, ["Low Engagement", "Developing", "Engaged", "Highly Engaged"]
    )
}

task24_df['Lifecycle_Stage'] = (
    task24_df["Engagement_Cluster"].map(lifecycle_mapping)
)

# 7. Display summary
print("Dataset shape:", task24_df.shape)

display(
    task24_df[
        [
            "Total_Page_Visits",
            "Total_Time_Spent",
            "Log_Total_Page_Visits",
            "Log_Total_Time_Spent",
            "Engagement_Score",
            "Lifecycle_Stage"
        ]
    ].head()
)

print("\nLifecycle stage distribution:")
display(
    task24_df["Lifecycle_Stage"]
    .value_counts()
    .rename_axis("Lifecycle Stage")
    .reset_index(name="Session Count")
)

print("\nAverage engagement by lifecycle stage:")
display(
    task24_df.groupby("Lifecycle_Stage")[
        ["Total_Page_Visits", "Total_Time_Spent", "Engagement_Score"]
    ].mean().round(3)
)

# 8. Save the result
task24_df.to_csv("task24_lifecycle_dataset.csv", index=False)

print("\nSaved: task24_lifecycle_dataset.csv")



Dataset shape: (12330, 27)


,Total_Page_Visits,Total_Time_Spent,Log_Total_Page_Visits,Log_Total_Time_Spent,Engagement_Score,Lifecycle_Stage
0,1,0.000000,0.693147,0.000000,0.052384,Low Engagement
1,2,64.000000,1.098612,4.174387,0.270132,Developing
2,1,0.000000,0.693147,0.000000,0.052384,Low Engagement
3,2,2.666667,1.098612,1.299283,0.141263,Low Engagement
4,10,627.500000,2.397895,6.443336,0.470023,Engaged



Lifecycle stage distribution:


,Lifecycle Stage,Session Count
0,Engaged,5205
1,Highly Engaged,3506
2,Developing,2801
3,Low Engagement,818



Average engagement by lifecycle stage:


,Total_Page_Visits,Total_Time_Spent,Engagement_Score
Lifecycle_Stage,,,
Developing,6.325,167.058,0.358
Engaged,21.639,817.788,0.523
Highly Engaged,83.974,3259.390,0.676
Low Engagement,1.518,1.003,0.077



Saved: task24_lifecycle_dataset.csv


### Task 24: Membership Duration and Lifecycle Segmentation

Engagement-related features were created using total page visits and total time spent on the website. Log transformation was applied to reduce the effect of highly skewed values, and Min-Max scaling was used to normalize the transformed features. K-Means clustering was then used to segment 12,330 website sessions into four engagement groups: Low Engagement, Developing, Engaged, and Highly Engaged.

The Engaged group contained the largest number of sessions (5,205), followed by Highly Engaged (3,506), Developing (2,801), and Low Engagement (818). The average engagement score increased across the groups, indicating meaningful separation by engagement level.

**Limitation:** The available dataset does not contain customer registration dates or membership start dates. Therefore, actual membership duration and true customer lifecycle stages could not be calculated. The resulting groups represent session engagement levels rather than verified customer membership stages.

Completed: Engagement features, log transformation, normalization, K-Means clustering, and 4 engagement stages.



Remaining: Actual membership duration calculation, because tamara dataset ma membership start date nathi.